# Représentation des données textuelles
## Nettoyage de texte

In [1]:
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

def netoyage(corpus_ensemble_documents):
    for i in range(len(corpus_ensemble_documents)):
        corpus_ensemble_documents[i]=corpus_ensemble_documents[i].lower()
    for i in range(len(corpus_ensemble_documents)):
        for c in string.punctuation:
            x=corpus_ensemble_documents[i].replace(c,' ')
            corpus_ensemble_documents[i]=x
    stopwords_anglais=stopwords.words('english') #ou french
    for i in range(len(corpus_ensemble_documents)):
        L=corpus_ensemble_documents[i].split()
        for mot in L:
            if mot in stopwords_anglais:
                while mot in L:
                    L.remove(mot)
        corpus_ensemble_documents[i]=" ".join(L)
    return(corpus_ensemble_documents)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mar1shell/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Calcul de TF et IDF

In [2]:
from numpy import *
def TF(terme,corpus,numero_document):
    x=corpus[numero_document].count(terme)
    y=len(corpus[numero_document].split())
    return x/y

def IDF(terme,corpus,numero_document):
    D=len(corpus)
    d_t=0
    for document in corpus:
        if terme in document:
            d_t+=1
    TF_val=TF(terme,corpus,numero_document)
    return TF_val*log(1+(D/d_t))

def cles_correspondante_a_valeur(valeur,dictionnaire):
    for cle in dictionnaire.keys():
        if dictionnaire[cle]==valeur:
            return cle

## Matrice Sparse

In [3]:
import numpy
def matrice_sparse(dictionnaire,corpus_ensemble_documents):
    M=numpy.zeros((len(corpus_ensemble_documents),len(dictionnaire.values())))
    for i in range(len(corpus_ensemble_documents)):
        for j in dictionnaire.values():
            x=cles_correspondante_a_valeur(j,dictionnaire)
            M[i,j]=IDF(x,corpus_ensemble_documents,i)
    return M

def affiche(M):
    (n,p)=M.shape
    for i in range(n):
        for j in range(p):
            M[i,j]=round(M[i,j],2)
    print(M)

## Test de la matrice sparse avec TfidfVectorizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
texte = ["La vie est douce", "La vie est tranquille, est belle, est douce",
         "le corona-virus est méchant"]
texte=netoyage(texte)
vect = TfidfVectorizer()
T= vect.fit_transform(texte)
dictionnaire_des_mots=vect.vocabulary_
print("dictionnaire_des_mots :", dictionnaire_des_mots)
liste_des_mots=list(dictionnaire_des_mots.keys())
print("liste_des_mots :", liste_des_mots)
Matrice_sparse_correspondante=T.toarray()
print("Matrice_sparse_methode predefinie:\n")
affiche(Matrice_sparse_correspondante)
#on donne un poid important aux tokens qui apparaissent souvent dans un
#document en particulier mais pas dans tous les documents du corpus
print("Matrice_sparse obtenue par notre methode:\n")
affiche(matrice_sparse(dictionnaire_des_mots,texte))

dictionnaire_des_mots : {'la': 4, 'vie': 8, 'est': 3, 'douce': 2, 'tranquille': 7, 'belle': 0, 'le': 5, 'corona': 1, 'virus': 9, 'méchant': 6}
liste_des_mots : ['la', 'vie', 'est', 'douce', 'tranquille', 'belle', 'le', 'corona', 'virus', 'méchant']
Matrice_sparse_methode predefinie:

[[0.   0.   0.53 0.41 0.53 0.   0.   0.   0.53 0.  ]
 [0.38 0.   0.29 0.68 0.29 0.   0.   0.38 0.29 0.  ]
 [0.   0.48 0.   0.28 0.   0.48 0.48 0.   0.   0.48]]
Matrice_sparse obtenue par notre methode:

[[0.   0.   0.23 0.17 0.23 0.   0.   0.   0.23 0.  ]
 [0.17 0.   0.11 0.26 0.11 0.23 0.   0.17 0.11 0.  ]
 [0.   0.28 0.   0.14 0.   0.18 0.28 0.   0.   0.28]]


# Application de la régression Logistique sur les Spams
## Sans binary=True

In [ ]:
import numpy as np
import pandas
from sklearn.model_selection import train_test_split

spams = pandas.read_table("SMSSpamCollection.txt",sep="\t",header=0)
spamsTrain, spamsTest = train_test_split(spams,train_size=0.7,random_state=1)
from sklearn.feature_extraction.text import CountVectorizer
parseur = CountVectorizer()
XTrain = parseur.fit_transform(spamsTrain['message'])
mdtTrain = XTrain.toarray()
from sklearn.linear_model import LogisticRegression
modelFirst= LogisticRegression()
#essayer cette version, pourquoi ça ne marche pas ?
#modelFirst.fit(spamsTrain['message'],spamsTrain['classe'])
modelFirst.fit(mdtTrain,spamsTrain['classe'])
score1=modelFirst.score(mdtTrain, spamsTrain['classe'])
print("score sur data train:"+str(score1))
mdtTest = parseur.transform(spamsTest['message'])
#predTest = modelFirst.predict(mdtTest)
score2=modelFirst.score(mdtTest, spamsTest['classe'])
print("score sur data test:"+str(score2))

score sur data train:0.997948717948718
score sur data test:0.9814593301435407


## Avec binary=True

In [6]:
import numpy as np
import pandas
spams = pandas.read_table("SMSSpamCollection.txt",sep="\t",header=0)
spamsTrain, spamsTest = train_test_split(spams,train_size=0.7,random_state=1)
from sklearn.feature_extraction.text import CountVectorizer
parseur = CountVectorizer(binary=True)
XTrain = parseur.fit_transform(spamsTrain['message'])
mdtTrain = XTrain.toarray()
from sklearn.linear_model import LogisticRegression
modelFirst= LogisticRegression()
#essayer cette version, pourquoi ça ne marche pas ?
#modelFirst.fit(spamsTrain['message'],spamsTrain['classe'])
modelFirst.fit(mdtTrain,spamsTrain['classe'])
score1=modelFirst.score(mdtTrain, spamsTrain['classe'])
print("score sur data train:"+str(score1))
mdtTest = parseur.transform(spamsTest['message'])
#predTest = modelFirst.predict(mdtTest)
score2=modelFirst.score(mdtTest, spamsTest['classe'])
print("score sur data test:"+str(score2))

score sur data train:0.9971794871794872
score sur data test:0.9844497607655502


## Sans stop_words

In [7]:
from sklearn import metrics
parseurBis = CountVectorizer(stop_words='english')
XTrainBis = parseurBis.fit_transform(spamsTrain['message'])
mdtTrainBis = XTrainBis.toarray()
modelBis = LogisticRegression()
modelBis.fit(mdtTrainBis,spamsTrain['classe'])
mdtTestBis = parseurBis.transform(spamsTest['message'])
score3=modelBis.score(mdtTrainBis,spamsTrain['classe'])
print("score sur data train:"+str(score3))
score4=modelBis.score(mdtTestBis, spamsTest['classe'])
print("score sur data test:"+str(score4))

score sur data train:0.9958974358974358
score sur data test:0.9766746411483254


## Avec TfidfVectorizer

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import metrics

parseur3 = TfidfVectorizer()
XTrain3 = parseur3.fit_transform(spamsTrain['message'])
mdtTrain3 = XTrain3.toarray()
model3 = LogisticRegression()
model3.fit(mdtTrain3,spamsTrain['classe'])
mdtTest3 = parseur3.transform(spamsTest['message'])
score5=model3.score(mdtTrain3,spamsTrain['classe'])
print("score sur data train:"+str(score5))
score6=model3.score(mdtTest3, spamsTest['classe'])
print("score sur data test:"+str(score6))

score sur data train:0.9743589743589743
score sur data test:0.9712918660287081
